# Bronze Ingestion — FHIR JSON Bundles

**Source:** Synthea-generated FHIR R4 JSON bundles (one file per patient)  
**Destination:** `clinical_lakehouse.bronze.fhir_bundle` (external Delta table)  
**Format:** Auto Loader incremental ingestion
**Layer:** Bronze — raw payload preserved, no transformation applied

## 1. Configuration

In [0]:
# Storage paths
STORAGE_ACCOUNT = "clinicallakeousestg"
BRONZE_CONTAINER = "bronze"

# Source path — where Synthea FHIR JSON files were uploaded in ADLS
SOURCE_PATH = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/synthea/fhir-json/"

# Delta table path — where the external table data will live in ADLS
DELTA_TABLE_PATH = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/fhir_bundle/"

# Auto Loader checkpoint — tracks which files have already been processed
CHECKPOINT_PATH = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/fhir_bundle/"

# Unity Catalog target
CATALOG = "clinical_lakehouse"
SCHEMA = "bronze"
TABLE_NAME = "fhir_bundle"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

print(f"Source path:      {SOURCE_PATH}")
print(f"Delta table path: {DELTA_TABLE_PATH}")
print(f"Checkpoint path:  {CHECKPOINT_PATH}")
print(f"Target table:     {FULL_TABLE_NAME}")

## 2. Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    ArrayType, BooleanType
)
import uuid

## 3. Verify Source Path Accessibility

In [0]:
try:
    files = dbutils.fs.ls(SOURCE_PATH)
    print(f"SUCCESS - Source path accessible")
    print(f"Files found: {len(files)}")
    for f in files[:5]:
        print(f"  {f.name} ({f.size / 1024:.1f} KB)")
    if len(files) > 5:
        print(f"  ... and {len(files) - 5} more files")
except Exception as e:
    print(f"ERROR - Cannot access source path: {e}")
    raise

## 4. Define FHIR Bundle Schema

Synthea FHIR bundles follow the R4 Bundle structure:
 ```
 {
   "resourceType": "Bundle",
   "type": "collection",
   "entry": [
     { "resource": { "resourceType": "Patient", ... } },
     { "resource": { "resourceType": "Encounter", ... } },
     ...
   ]
 }
 ```
 At bronze we store the full raw payload and extract only
 lightweight summary fields for partitioning and filtering.

In [0]:
# Define the schema for each entry's resource
# Using MapType(StringType(), StringType()) at bronze since we only
# need resourceType and id — full parsing happens in Silver
resource_schema = StructType([
    StructField("resourceType", StringType(), True),
    StructField("id", StringType(), True)
])

# Define the schema for each entry in the array
entry_schema = ArrayType(
    StructType([
        StructField("fullUrl", StringType(), True),
        StructField("resource", resource_schema, True),
        StructField("request", StructType([
            StructField("method", StringType(), True),
            StructField("url", StringType(), True)
        ]), True)
    ])
)

print("Entry schema defined successfully")
print(entry_schema.simpleString())

## 5. Auto Loader — Read FHIR JSON Files

Auto Loader tracks which files have been processed via the checkpoint.
Re-running this notebook will only process **new** files added since
the last run — already ingested files are skipped automatically.

In [0]:
# Read FHIR JSON bundles using Auto Loader
raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "schema/")
    # FHIR bundles are multi-line JSON — one bundle spans multiple lines
    .option("multiLine", "true")
    # Allow schema to evolve if new FHIR resource types appear in future runs
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    # Capture the source file path for lineage tracking
    .option("cloudFiles.includeExistingFiles", "true")
    .load(SOURCE_PATH)
)

## 6. Add Bronze Metadata Columns

Every bronze table gets these standard metadata columns:
- `_source_file_path` — exact ADLS path of the source file
- `_source_file_name` — just the filename for readability  
- `_source_file_modification_time` — when the file was last modified in ADLS
- `_ingestion_timestamp` — when this row was written to Delta
- `_batch_id` — unique ID per ingestion run for debugging and reprocessing

In [0]:
# Generate a unique batch ID for this ingestion run
batch_id = str(uuid.uuid4())
print(f"Batch ID for this run: {batch_id}")

In [0]:
# Build the bronze DataFrame with metadata columns
bronze_df = (
    raw_df

    # ── Core bundle fields ────────────────────────────────────────────────────
    .withColumn("bundle_resource_type", F.col("resourceType"))
    .withColumn("bundle_type", F.col("type"))

    # ── Parse entry from STRING → ARRAY using explicit schema ────────────────
    # Auto Loader inferred entry as STRING — we cast it to the proper
    # array structure so we can use array functions on it
    .withColumn(
        "entry_parsed",
        F.from_json(F.col("entry"), entry_schema)
    )

    # ── Entry count ───────────────────────────────────────────────────────────
    .withColumn("entry_count", F.size(F.col("entry_parsed")))

    # ── Extract patient ID from the parsed entry array ────────────────────────
    .withColumn(
        "patient_id",
        F.expr("""
            filter(entry_parsed, 
                e -> e.resource.resourceType = 'Patient'
            )[0].resource.id
        """)
    )

    # ── Resource type summary ─────────────────────────────────────────────────
    .withColumn(
        "resource_type_summary",
        F.expr("transform(entry_parsed, e -> e.resource.resourceType)")
    )

    # ── Raw bundle — preserve full original JSON for Silver parsing ───────────
    # We store the original entry string column (before parsing) as part
    # of the raw payload so Silver gets the complete unmodified JSON
    .withColumn("raw_bundle", F.to_json(F.struct(
        F.col("resourceType").alias("resourceType"),
        F.col("type").alias("type"),
        F.col("entry").alias("entry")
    )))

    # ── Bronze metadata columns ───────────────────────────────────────────────
    .withColumn("_source_file_path", F.input_file_name())
    .withColumn(
        "_source_file_name",
        F.element_at(F.split(F.input_file_name(), "/"), -1)
    )
    .withColumn(
        "_source_file_modification_time",
        F.col("_metadata.file_modification_time")
    )
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_batch_id", F.lit(batch_id))

    # ── Partition column ──────────────────────────────────────────────────────
    .withColumn("_ingestion_date", F.to_date(F.col("_ingestion_timestamp")))

    # ── Drop staging columns ──────────────────────────────────────────────────
    .drop(
        "resourceType", 
        "type", 
        "entry",          # original string version — preserved in raw_bundle
        "entry_parsed",   # working column — data captured in summary columns
        "_metadata", 
        "_rescued_data"
    )
)

print("bronze_df defined successfully")
bronze_df.printSchema()

In [0]:
preview_df = (
    spark.read
    .format("json")
    .option("multiLine", "true")
    .load(SOURCE_PATH)
    .limit(3)

    # ── Core bundle fields ────────────────────────────────────────────────────
    # In batch read, entry is already a proper ARRAY — no from_json needed
    .withColumn("bundle_resource_type", F.col("resourceType"))
    .withColumn("bundle_type", F.col("type"))

    # ── Entry count ───────────────────────────────────────────────────────────
    .withColumn("entry_count", F.size(F.col("entry")))

    # ── Extract patient ID directly using dot notation ────────────────────────
    # Batch read parses the full nested structure so we access fields directly
    .withColumn(
        "patient_id",
        F.expr("""
            filter(entry,
                e -> e.resource.resourceType = 'Patient'
            )[0].resource.id
        """)
    )

    # ── Resource type summary ─────────────────────────────────────────────────
    .withColumn(
        "resource_type_summary",
        F.expr("transform(entry, e -> e.resource.resourceType)")
    )

    # ── Source file name ──────────────────────────────────────────────────────
    .withColumn(
        "_source_file_name",
        F.element_at(F.split(F.input_file_name(), "/"), -1)
    )

    # ── Select only display columns ───────────────────────────────────────────
    .select(
        "bundle_resource_type",
        "bundle_type",
        "patient_id",
        "entry_count",
        "resource_type_summary",
        "_source_file_name"
    )
)

display(preview_df)

## 8. Write to Bronze External Delta Table

Writing as an **external** table because:
- We explicitly define the ADLS path so file layout is visible
- Dropping the table does not delete underlying files
- Raw data is preserved independent of the Databricks table lifecycle
- Matches production healthcare DE pattern for raw data preservation

In [0]:
(
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    # Checkpoint ensures Auto Loader remembers which files were
    # already processed — re-running will only pick up new files
    .option("checkpointLocation", CHECKPOINT_PATH)
    # Explicit path registers this as an external table
    .option("path", DELTA_TABLE_PATH)
    # Partition by ingestion date for efficient downstream reads
    .partitionBy("_ingestion_date")
    # availableNow processes all existing files then stops cleanly
    # This replaces the old time.sleep() workaround — more reliable
    # and deterministic for notebook development and scheduled jobs
    .trigger(availableNow=True)
    # Register in Unity Catalog
    .toTable(FULL_TABLE_NAME)
    # Block notebook execution until stream finishes
    .awaitTermination()
)

print(f"Stream completed — all available files processed")

## 9. Validate — Row Counts and Sample Data


In [0]:
# Total row count — should equal number of FHIR JSON files uploaded
row_count = spark.table(FULL_TABLE_NAME).count()
print(f"Total rows in {FULL_TABLE_NAME}: {row_count:,}")
print(f"Expected: one row per Synthea patient FHIR bundle")

In [0]:
# Sample records to verify all columns populated correctly
display(
    spark.table(FULL_TABLE_NAME)
    .select(
        "patient_id",
        "bundle_type",
        "entry_count",
        "resource_type_summary",
        "_source_file_name",
        "_ingestion_timestamp",
        "_batch_id",
        "_ingestion_date"
    )
    .limit(5)
)

In [0]:
# Verify resource type distribution across all bundles
# This confirms what clinical entities are available for Silver parsing
display(
    spark.table(FULL_TABLE_NAME)
    .select(
        F.explode("resource_type_summary").alias("resource_type")
    )
    .groupBy("resource_type")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
# Confirm table is registered as EXTERNAL not MANAGED
# Look for "Type = EXTERNAL" in the output
spark.sql(f"DESCRIBE EXTENDED {FULL_TABLE_NAME}").display()

In [0]:
# Confirm Delta table properties and physical file location in ADLS
spark.sql(f"DESCRIBE DETAIL {FULL_TABLE_NAME}").display()

In [0]:
%sql
SELECT * FROM clinical_lakehouse.bronze.fhir_bundle LIMIT 100;

In [0]:
# Total row count should now reflect the additional patients
display(
    spark.table(FULL_TABLE_NAME)
    .groupBy("_ingestion_date", "_batch_id")
    .count()
    .orderBy(F.desc("_ingestion_date"))
)